In [ ]:
pip install --upgrade --force-reinstall pandas scikit-learn


In [ ]:

import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
df = pd.read_csv(r"C:\Users\gunja\Desktop\Tecnomers internship\systematic lit review\bibliometric\18_final_article_csv.csv", encoding="ISO-8859-1")
# Clean columns
df["Author"] = df["Author"].fillna("").str.replace(",", ";")
df["Author Keyword"] = df["Author Keyword"].fillna("").str.replace(",", ";")
df["Title"] = df["Title"].fillna("").astype(str)
# === 1. Save co-authorship data ===
df["Author"].to_csv("vosviewer_authors.txt", index=False, header=False)

# === 2. Save keyword co-occurrence data ===
df["Author Keyword"].to_csv("vosviewer_keywords.txt", index=False, header=False)

# === 3. Extract term co-occurrence from titles ===
def clean_text(text):
    # Remove punctuation and numbers
    return re.sub(r"[^\w\s]", "", text.lower())
# Clean titles
titles = df["Title"].apply(clean_text).tolist()
# Vectorize to extract frequent terms
vectorizer = CountVectorizer(stop_words='english', min_df=2)  # Adjust min_df to control frequency
X = vectorizer.fit_transform(titles)
terms = vectorizer.get_feature_names_out()
doc_term_matrix = X.toarray()
# Convert matrix to VOSviewer format: one line per document, terms separated by ;
term_lines = []
for row in doc_term_matrix:
    words = [terms[i] for i, val in enumerate(row) if val > 0]
    term_lines.append("; ".join(words))
# Save to file
with open("vosviewer_terms.txt", "w", encoding="utf-8") as f:
    for line in term_lines:
        f.write(line + "\n")


C:\Users\gunja\AppData\Local\Temp\ipykernel_38480\3665964582.py:5: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\gunja\Desktop\Tecnomers internship\systematic lit review\bibliometric\18_final_article_csv.csv", encoding="ISO-8859-1")


In [48]:
df = pd.read_csv(r"C:\Users\gunja\Desktop\Tecnomers internship\systematic lit review\bibliometric\71_Final_full_atricles - Copy.csv", encoding="ISO-8859-1")
df.head()

,Authors,Title,DOI,Link,Author Keywords,Abstract
0,Echefu G.; Shah R.; Sanchez Z.; Rickards J.; B...,Artificial intelligence: Applications in cardi...,10.1016/j.ahjo.2024.100479,https://www.scopus.com/inward/record.uri?eid=2...,Artificial intelligence; Cancer; Cardio-oncolo...,Numerous cancer therapies have detrimental car...
1,Cong R.; Deng O.; Nishimura S.; Ogihara A.; Ji...,Multiple feature selection based on an optimiz...,10.1007/s13755-024-00312-8,https://www.scopus.com/inward/record.uri?eid=2...,Causal graphs; Feature selection; Health data ...,Purpose : Recent advancements in information t...
2,Salim A.; Brakenridge C.J.; Lekamlage D.H.; Ho...,Detection of sedentary time and bouts using co...,10.1186/s12874-024-02311-5,https://www.scopus.com/inward/record.uri?eid=2...,Bouts; Heart rate; Machine learning; Step coun...,Background: Wrist-worn data from commercially ...
3,Chen W.; Cordero R.; Lever Taylor J.; Pangallo...,Multicenter Evaluation of Machine-Learning Con...,10.1159/000542615,https://www.scopus.com/inward/record.uri?eid=2...,Digital health technologies; Machine learning;...,Introduction: Though wrist-worn photoplethysmo...
4,Azhibekova Z.; Altayeva A.; Amirtayeva A.; Mom...,Advancing athlete safety through real-time ECG...,10.47197/retos.v61.110378,https://www.scopus.com/inward/record.uri?eid=2...,athlete cardiovascular health; physiological m...,This research paper explores the implementatio...


In [4]:
# Clean columns
df["Author"] = df["Author"].fillna("").str.replace(",", ";")
df["Author Keyword"] = df["Author Keyword"].fillna("").str.replace(",", ";")
df["Title"] = df["Title"].fillna("").astype(str)

In [5]:
# === 1. Save co-authorship data ===
df["Author"].to_csv("vosviewer_authors.txt", index=False, header=False)

# === 2. Save keyword co-occurrence data ===
df["Author Keyword"].to_csv("vosviewer_keywords.txt", index=False, header=False)

# === 3. Extract term co-occurrence from titles ===
def clean_text(text):
    # Remove punctuation and numbers
    return re.sub(r"[^\w\s]", "", text.lower())

In [6]:
# Clean titles
titles = df["Title"].apply(clean_text).tolist()

# Vectorize to extract frequent terms
vectorizer = CountVectorizer(stop_words='english', min_df=2)  # Adjust min_df to control frequency
X = vectorizer.fit_transform(titles)
terms = vectorizer.get_feature_names_out()
doc_term_matrix = X.toarray()

In [7]:
# Convert matrix to VOSviewer format: one line per document, terms separated by ;
term_lines = []
for row in doc_term_matrix:
    words = [terms[i] for i, val in enumerate(row) if val > 0]
    term_lines.append("; ".join(words))

# Save to file
with open("vosviewer_terms.txt", "w", encoding="utf-8") as f:
    for line in term_lines:
        f.write(line + "\n")

In [8]:
# === Step 2: Preprocess Abstracts ===
df['Abstract'] = df['Abstract'].fillna('').astype(str)

def clean_text(text):
    # Lowercase and remove special characters
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text

df['Cleaned_Abstract'] = df['Abstract'].apply(clean_text)

# === Step 3: Extract frequent terms using CountVectorizer ===
vectorizer = CountVectorizer(stop_words='english', min_df=2)  # min_df=2 filters rare words
X = vectorizer.fit_transform(df['Cleaned_Abstract'])
terms = vectorizer.get_feature_names_out()

# === Step 4: Build VOSviewer-style term list per document ===
doc_term_matrix = X.toarray()

term_lines = []
for row in doc_term_matrix:
    words = [terms[i] for i, val in enumerate(row) if val > 0]
    term_lines.append("; ".join(words))

# === Step 5: Save to VOSviewer-readable file ===
with open("vosviewer_abstract_terms.txt", "w", encoding="utf-8") as f:
    for line in term_lines:
        f.write(line + "\n")

In [58]:
import numpy as np
from itertools import combinations
from collections import Counter, defaultdict
# === 3. Create term co-occurrence network ===
term_occurrences = np.array(X.sum(axis=0)).flatten()
term_index = {term: idx for idx, term in enumerate(terms)}
index_term = {idx: term for term, idx in term_index.items()}

co_occurrence = defaultdict(int)
for doc_vec in X.toarray():
    present_terms = [i for i, v in enumerate(doc_vec) if v > 0]
    for i, j in combinations(present_terms, 2):
        pair = tuple(sorted((i, j)))
        co_occurrence[pair] += 1

# === 4. Save network.txt ===
with open("abstract_network.txt", "w", encoding="utf-8") as f:
    f.write("Source\tTarget\tWeight\n")
    for (i, j), weight in co_occurrence.items():
        f.write(f"{index_term[i]}\t{index_term[j]}\t{weight}\n")

# === 5. Save map.txt ===
with open("abstract_map.txt", "w", encoding="utf-8") as f:
    f.write("Label\tID\tWeight\n")
    for i, term in index_term.items():
        f.write(f"{term}\t{i}\t{term_occurrences[i]}\n")
